## Organismos selecionados

Os organismos foram selecionados a partir de:

- **Millán Arias et al. (2023)**: Supplementary Table S1, que reúne ~700 genomas procarióticos com temperatura ótima de crescimento (OGT) conhecida. Quatro dos seis organismos foram retirados diretamente dessa tabela.
- **Literatura de extremófilos**: Dois organismos foram adicionados para garantir representação de dois filos distintos em todas as categorias térmicas.

O critério de seleção foi garantir dois organismos de filos diferentes para cada categoria térmica (hipertermófilo, mesófilo e psicrófilo), permitindo avaliar se o agrupamento filogenético reflete o ambiente compartilhado ou o parentesco evolutivo.

| Organismo | Filo | Categoria | OGT | Assembly NCBI | 
|---|---|---|---|---|
| *Thermocrinis ruber* | Aquificae | Hipertermófilo | >80°C | GCA_000512735.1 
| *Thermotoga maritima* | Thermotogae | Hipertermófilo | >80°C | GCA_000008545.1 
| *Escherichia fergusonii* | Proteobacteria | Mesófilo | 20–45°C | GCA_000026225.1 
| *Bacillus subtilis* | Firmicutes | Mesófilo | 25–35°C | GCA_000009045.1 
| *Psychrobacter arcticus* | Proteobacteria | Psicrófilo | <20°C | GCA_000012305.1 
| *Exiguobacterium sibiricum* | Firmicutes | Psicrófilo | −5°C | GCA_000026245.1 

In [9]:
from Bio import AlignIO, SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
import requests
import os
import time
import subprocess


## 1. Obtenção das sequências nucleotídicas via KEGG

Após a identificação dos genes, as sequências nucleotídicas codificantes (CDS) foram obtidas diretamente por meio da API REST do KEGG. Para cada gene foi realizada uma requisição utilizando o identificador KEGG correspondente:

```text
https://rest.kegg.jp/get/<gene_id>/ntseq
```

Por exemplo, para o gene *deaD* de *Thermocrinis ruber*:

```text
https://rest.kegg.jp/get/trd:THERU_06610/ntseq
```

Esse endpoint retorna diretamente a sequência nucleotídica codificante associada ao gene

In [2]:
genes_kegg = {
    "T_ruber": {
        "dnaK":  "trd:THERU_01435",
        "groEL": "trd:THERU_06905",
        "gyrA":  "trd:THERU_01910",
        "deaD":  "trd:THERU_06610",
    },
    "T_maritima": {
        "dnaK":  "tma:TM0373",
        "groEL": "tma:TM0506",
        "gyrA":  "tma:TM1084",
        "deaD":  None,
    },
    "E_fergusonii": {
        "dnaK":  "efe:EFER_0010",
        "groEL": "efe:EFER_4195",
        "gyrA":  "efe:EFER_0934",
        "deaD":  "efe:EFER_3141",
    },
    "B_subtilis": {
        "dnaK":  "bsu:BSU25470",
        "groEL": "bsu:BSU06030",
        "gyrA":  "bsu:BSU00070",
        "deaD":  "bsu:BSU04580",
    },
    "P_arcticus": {
        "dnaK":  "par:Psyc_2132",
        "groEL": "par:Psyc_0553",
        "gyrA":  "par:Psyc_1543",
        "deaD":  None,
    },
    "E_sibiricum": {
        "dnaK":  "esi:Exig_0781",
        "groEL": "esi:Exig_2768",
        "gyrA":  "esi:Exig_0006",
        "deaD":  "esi:Exig_0618",
    },
}

In [4]:
def baixar_ntseq(kegg_id):
    """Baixa a sequência nucleotídica de um gene via API REST do KEGG."""
    url = f"https://rest.kegg.jp/get/{kegg_id}/ntseq"
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"Erro HTTP {r.status_code} para {kegg_id}")
    texto = r.text.strip()
    if not texto.startswith(">"):
        raise RuntimeError(f"Resposta inesperada para {kegg_id}")
    return texto


os.makedirs("sequencias_nt_funcionais", exist_ok=True)

por_gene = {"dnaK": [], "groEL": [], "gyrA": [], "deaD": []}
falhas = []

print("=" * 60)
print("Baixando sequências nucleotídicas via KEGG REST API")
print("=" * 60)

for organismo, genes in genes_kegg.items():
    print(f"\n── {organismo}")
    for gene, kegg_id in genes.items():
        if kegg_id is None:
            print(f"  {gene}: ausente")
            continue
        print(f"  {gene} ({kegg_id})... ", end="", flush=True)
        try:
            fasta = baixar_ntseq(kegg_id)
            linhas = fasta.splitlines()
            sequencia = "".join(l.strip() for l in linhas if not l.startswith(">"))
            fasta_final = f">{organismo}_{gene}\n{sequencia}"
            por_gene[gene].append(fasta_final)
            print("ok")
        except Exception as e:
            print(f"{e}")
            falhas.append((organismo, gene, str(e)))
        time.sleep(0.5)

print("\n" + "=" * 60)
print("Salvando arquivos FASTA")
print("=" * 60)

for gene, seqs in por_gene.items():
    if not seqs:
        continue
    arquivo = f"sequencias_nt_funcionais/{gene}_sequencias.fasta"
    with open(arquivo, "w") as f:
        f.write("\n\n".join(seqs))
    print(f"  {gene}: {len(seqs)} sequências → {arquivo}")

if falhas:
    print("\nFalhas:")
    for org, gene, erro in falhas:
        print(f"  {org} / {gene}: {erro}")

Baixando sequências nucleotídicas via KEGG REST API

── T_ruber
  dnaK (trd:THERU_01435)... 

ok
  groEL (trd:THERU_06905)... ok
  gyrA (trd:THERU_01910)... ok
  deaD (trd:THERU_06610)... ok

── T_maritima
  dnaK (tma:TM0373)... ok
  groEL (tma:TM0506)... ok
  gyrA (tma:TM1084)... ok
  deaD: ausente

── E_fergusonii
  dnaK (efe:EFER_0010)... ok
  groEL (efe:EFER_4195)... ok
  gyrA (efe:EFER_0934)... ok
  deaD (efe:EFER_3141)... ok

── B_subtilis
  dnaK (bsu:BSU25470)... ok
  groEL (bsu:BSU06030)... ok
  gyrA (bsu:BSU00070)... ok
  deaD (bsu:BSU04580)... ok

── P_arcticus
  dnaK (par:Psyc_2132)... ok
  groEL (par:Psyc_0553)... ok
  gyrA (par:Psyc_1543)... ok
  deaD: ausente

── E_sibiricum
  dnaK (esi:Exig_0781)... ok
  groEL (esi:Exig_2768)... ok
  gyrA (esi:Exig_0006)... ok
  deaD (esi:Exig_0618)... ok

Salvando arquivos FASTA
  dnaK: 6 sequências → sequencias_nt_funcionais/dnaK_sequencias.fasta
  groEL: 6 sequências → sequencias_nt_funcionais/groEL_sequencias.fasta
  gyrA: 6 sequências → sequencias_nt_funcionais/gyrA_sequencias.fasta
  deaD: 4 sequências → sequencias_nt_funci

## 2. Alinhamento múltiplo com MAFFT

Cada gene é alinhado separadamente com MAFFT usando a opção `--auto`, que seleciona automaticamente o algoritmo mais adequado para o conjunto de sequências 

O deaD é alinhado com 5 organismos (T. maritima e P. arcticus ausentes).

In [ ]:
#alinhamento 16s
from Bio import SeqIO
import subprocess
import os

os.makedirs("alinhamentos", exist_ok=True)

# Mapeamento accession → (organismo, ambiente, filo)
info_organismos = {
    "KF387705.1": ("E_sibiricum",  "Psicrofilo",     "Firmicutes"),
    "AY444823.1": ("P_arcticus",   "Psicrofilo",     "Proteobacteria"),
    "AB055007.1": ("B_subtilis",   "Mesofilo",       "Firmicutes"),
    "AB681775.1": ("E_fergusonii", "Mesofilo",       "Proteobacteria"),
    "AJ401017.1": ("T_maritima",   "Hipertermofilo", "Thermotogae"),
    "AJ005640.1": ("T_ruber",      "Hipertermofilo", "Aquificae"),
}

pasta_16S = "16S_sequences"
entrada   = "16S_sequences/16S_sequencias.fasta"
saida     = "alinhamentos/16S_alinhado.fasta"

# Combinar e renomear os headers com organismo_ambiente_filo
registros = []
for arquivo in sorted(os.listdir(pasta_16S)):
    if not arquivo.endswith(".fasta"):
        continue
    accession = arquivo.replace(".fasta", "")
    organismo, ambiente, filo = info_organismos.get(
        accession, (accession, "Desconhecido", "Desconhecido")
    )

    rec = SeqIO.read(os.path.join(pasta_16S, arquivo), "fasta")
    rec.id = f"{organismo}_{ambiente}_{filo}_16S"
    rec.description = ""
    registros.append(rec)

SeqIO.write(registros, entrada, "fasta")
print(f"Combinado: {len(registros)} sequências → {entrada}")
for rec in registros:
    print(f"  {rec.id}")
print()

# Rodar MAFFT
resultado = subprocess.run(
    ["mafft", "--auto", entrada],
    capture_output=True,
    text=True
)

if resultado.returncode != 0:
    print(f"✗ Erro:\n{resultado.stderr}")
else:
    with open(saida, "w") as f:
        f.write(resultado.stdout)

    # Extrair informações relevantes do log
    linhas = resultado.stderr.split("\n")

    algoritmo = next((l.strip() for l in linhas if "L-INS-i" in l or "FFT-NS" in l or "G-INS-i" in l), "—")
    modelo    = next((l.strip() for l in linhas if "model=" in l and "alg=L" in l), "")
    modelo    = modelo.split("model=")[1].split(",")[0].strip() if "model=" in modelo else "—"
    gap       = next((l.strip() for l in linhas if "Gap Penalty" in l), "—")
    n_seqs    = resultado.stdout.count(">")

    print(f"  16S")
    print(f"    Sequências:  {n_seqs}")
    print(f"    Algoritmo:   {algoritmo}")
    print(f"    Modelo:      {modelo}")
    print(f"    {gap}")
    print(f"    Saída:       {saida}")

Combinado: 6 sequências → 16S_sequences/16S_sequencias.fasta
  B_subtilis_Mesofilo_Firmicutes_16S
  E_fergusonii_Mesofilo_Proteobacteria_16S
  T_ruber_Hipertermofilo_Aquificae_16S
  T_maritima_Hipertermofilo_Thermotogae_16S
  P_arcticus_Psicrofilo_Proteobacteria_16S
  E_sibiricum_Psicrofilo_Firmicutes_16S

  16S
    Sequências:  6
    Algoritmo:   L-INS-i (Probably most accurate, very slow)
    Modelo:      DNA200 (2)
    Gap Penalty = -1.53, +0.00, +0.00
    Saída:       alinhamentos/16S_alinhado.fasta


In [6]:
from Bio import SeqIO
import subprocess
import os

os.makedirs("alinhamentos", exist_ok=True)

# Mapeamento organismo → (ambiente, filo)
info_organismos = {
    "T_ruber":      ("Hipertermofilo", "Aquificae"),
    "T_maritima":   ("Hipertermofilo", "Thermotogae"),
    "E_fergusonii": ("Mesofilo",       "Proteobacteria"),
    "B_subtilis":   ("Mesofilo",       "Firmicutes"),
    "P_arcticus":   ("Psicrofilo",     "Proteobacteria"),
    "E_sibiricum":  ("Psicrofilo",     "Firmicutes"),
}

genes = ["dnaK", "groEL", "gyrA", "deaD"]

print("Rodando MAFFT...\n")

for gene in genes:
    entrada_original = f"sequencias_nt_funcionais/{gene}_sequencias.fasta"
    entrada_renomeada = f"sequencias_nt_funcionais/{gene}_renomeado.fasta"
    saida   = f"alinhamentos/{gene}_alinhado.fasta"

    if not os.path.exists(entrada_original):
        print(f"  {gene}: arquivo não encontrado")
        continue

    # Renomear headers para organismo_ambiente_filo_gene
    registros = []
    for rec in SeqIO.parse(entrada_original, "fasta"):
        organismo = rec.id.replace(f"_{gene}", "")
        ambiente, filo = info_organismos.get(organismo, ("Desconhecido", "Desconhecido"))
        rec.id = f"{organismo}_{ambiente}_{filo}_{gene}"
        rec.description = ""
        registros.append(rec)

    SeqIO.write(registros, entrada_renomeada, "fasta")

    resultado = subprocess.run(
        ["mafft", "--auto", entrada_renomeada],
        capture_output=True,
        text=True
    )

    if resultado.returncode != 0:
        print(f"  {gene}: Erro\n{resultado.stderr}")
        continue

    with open(saida, "w") as f:
        f.write(resultado.stdout)

    # Extrair informações relevantes do log
    linhas = resultado.stderr.split("\n")

    algoritmo = next((l.strip() for l in linhas if "L-INS-i" in l or "FFT-NS" in l or "G-INS-i" in l), "—")
    modelo    = next((l.strip() for l in linhas if "model=" in l and "alg=L" in l), "")
    modelo    = modelo.split("model=")[1].split(",")[0].strip() if "model=" in modelo else "—"
    gap       = next((l.strip() for l in linhas if "Gap Penalty" in l), "—")
    n_seqs    = resultado.stdout.count(">")

    print(f"  {gene}")
    print(f"    Sequências:  {n_seqs}")
    print(f"    Algoritmo:   {algoritmo}")
    print(f"    Modelo:      {modelo}")
    print(f"    {gap}")
    print(f"    Saída:       {saida}")
    print()

print("Concluído.")

Rodando MAFFT...

  dnaK
    Sequências:  6
    Algoritmo:   L-INS-i (Probably most accurate, very slow)
    Modelo:      DNA200 (2)
    Gap Penalty = -1.53, +0.00, +0.00
    Saída:       alinhamentos/dnaK_alinhado.fasta

  groEL
    Sequências:  6
    Algoritmo:   L-INS-i (Probably most accurate, very slow)
    Modelo:      DNA200 (2)
    Gap Penalty = -1.53, +0.00, +0.00
    Saída:       alinhamentos/groEL_alinhado.fasta

  gyrA
    Sequências:  6
    Algoritmo:   L-INS-i (Probably most accurate, very slow)
    Modelo:      DNA200 (2)
    Gap Penalty = -1.53, +0.00, +0.00
    Saída:       alinhamentos/gyrA_alinhado.fasta

  deaD
    Sequências:  4
    Algoritmo:   L-INS-i (Probably most accurate, very slow)
    Modelo:      DNA200 (2)
    Gap Penalty = -1.53, +0.00, +0.00
    Saída:       alinhamentos/deaD_alinhado.fasta

Concluído.


## 3. Verificação dos alinhamentos

Conferir número de sequências e comprimento do alinhamento para cada gene.

In [7]:

print(f"{'Gene':<8} {'Sequências':>12} {'Comprimento (bp)':>18}")
print("-" * 42)

for gene in genes:
    arquivo = f"alinhamentos/{gene}_alinhado.fasta"
    if not os.path.exists(arquivo):
        print(f"{gene:<8} {'—':>12} {'—':>18}")
        continue
    aln = AlignIO.read(arquivo, "fasta")
    print(f"{gene:<8} {len(aln):>12} {aln.get_alignment_length():>18}")



Gene       Sequências   Comprimento (bp)
------------------------------------------
dnaK                6               1995
groEL               6               1662
gyrA                6               2913
deaD                4               1990


## 4. Concatenação para a árvore MLSA

Os alinhamentos de dnaK, groEL e gyrA são concatenados em uma sequência única por organismo.
O deaD é excluído da concatenação por estar ausente em T. maritima e P. arcticus.

Metodologia baseada em Bouvet et al. (2014), que usaram o software START2 para concatenar 6 genes funcionais antes de construir a árvore MLSA.

In [10]:
genes_mlsa = ["dnaK", "groEL", "gyrA"]

# Carregar alinhamentos removendo sufixo do gene do ID
alinhamentos = {}
for gene in genes_mlsa:
    aln = AlignIO.read(f"alinhamentos/{gene}_alinhado.fasta", "fasta")
    alinhamentos[gene] = {
        rec.id.replace(f"_{gene}", ""): str(rec.seq)
        for rec in aln
    }

# Organismos presentes em todos os genes
organismos = set(alinhamentos[genes_mlsa[0]].keys())
for gene in genes_mlsa[1:]:
    organismos &= set(alinhamentos[gene].keys())

# Concatenar
concatenadas = []
for org in sorted(organismos):
    seq_concat = "".join(alinhamentos[gene][org] for gene in genes_mlsa)
    concatenadas.append(SeqRecord(Seq(seq_concat), id=org, description=""))

SeqIO.write(concatenadas, "alinhamentos/concatenado_mlsa.fasta", "fasta")

# Resumo
comprimentos = {gene: aln.get_alignment_length()
                for gene, aln in
                [(g, AlignIO.read(f"alinhamentos/{g}_alinhado.fasta", "fasta"))
                 for g in genes_mlsa]}
total = sum(comprimentos.values())

print("Alinhamentos individuais:")
print(f"  dnaK:  {comprimentos['dnaK']} bp")
print(f"  groEL: {comprimentos['groEL']} bp")
print(f"  gyrA:  {comprimentos['gyrA']} bp")
print(f"  Total: {comprimentos['dnaK']} + {comprimentos['groEL']} + {comprimentos['gyrA']} = {total} bp")
print(f"\nOrganismos na concatenada: {len(concatenadas)}")
for rec in concatenadas:
    print(f"  {rec.id}  ({len(rec.seq)} bp)")
print(f"\nSalvo: alinhamentos/concatenado_mlsa.fasta")

Alinhamentos individuais:
  dnaK:  1995 bp
  groEL: 1662 bp
  gyrA:  2913 bp
  Total: 1995 + 1662 + 2913 = 6570 bp

Organismos na concatenada: 6
  B_subtilis_Mesofilo_Firmicutes  (6570 bp)
  E_fergusonii_Mesofilo_Proteobacteria  (6570 bp)
  E_sibiricum_Psicrofilo_Firmicutes  (6570 bp)
  P_arcticus_Psicrofilo_Proteobacteria  (6570 bp)
  T_maritima_Hipertermofilo_Thermotogae  (6570 bp)
  T_ruber_Hipertermofilo_Aquificae  (6570 bp)

Salvo: alinhamentos/concatenado_mlsa.fasta
